# Dataset Profiling and Scenario Mapping

Three approaches to profiling and matching real datasets to L9 simulation scenarios.

**Notebook placement:** put this file at your **real dataset root** — the same level as your existing `profiling.ipynb`. The L9 path is resolved relative to that location.

```
Real_Dataset_Root/               ← notebook lives here
  Study_A/
    for_use/
      matrix.mtx
      barcodes.tsv
      genes.tsv          (or features.tsv)
      metadata.csv
    original/ ...
  Study_B/ ...

../P4 L9 Synthetic V1/           ← one level up, sibling directory
  load_to_anndata.ipynb
  l9_benchmark_sets/
    S1/
      rep1/
        counts.mtx
        genes.txt
        cells.txt
        metadata.csv     ('cell_type' column)
      rep2/ rep3/
    S2/ ... S9/
```

---

## On raw vs. preprocessed inputs for profiling

| Metric group | Correct input | Rationale |
|---|---|---|
| `Dropout_Pct`, `Median_UMI_Per_Cell` | Raw counts | After CP10K all cells sum to 10,000 — depth metrics become meaningless |
| `BCV_Dispersion_Proxy` | Raw counts | Overdispersion is a count-data property |
| `Silhouette_Score`, `KNN_Purity` | Log-norm + HVG + PCA | Geometry should reflect biological variation, not library size |
| `Mean_Pi_Score` | Log-normalised (pre-scale) | Wilcoxon LFC on raw counts conflates depth with signal |
| `Norm_Shannon_Entropy`, `Min_Cluster_Size`, `Total_CellTypes` | Cell type labels only | Expression-independent |

The `profile_dataset()` function loads raw counts and internally applies the correct representation per metric group.

---

## Section overview
- **Section 1** — Profile all datasets, save CSVs, PCA + matching with **10 metrics**
- **Section 2** — Load CSVs, PCA + matching with **4 L9 design axes only**
- **Section 3** — Load CSVs, fit PCA on **real data only**, project L9 into that space, match in PC space

---
## 0. Imports, Paths, and Shared Helpers
**Run this cell before any section.**

In [ ]:
import gc
import warnings
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from scipy.io import mmread
from scipy.stats import entropy
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

warnings.filterwarnings('ignore')
sc.settings.verbosity = 0

# =========================================================================
# PATHS
# Notebook lives at real dataset root.
# L9 lives one level up inside 'P4 L9 Synthetic V1/l9_benchmark_sets'.
# =========================================================================
REAL_BASE_DIR = Path('.')
L9_BASE_DIR   = Path('../P4 L9 Synthetic V1/l9_benchmark_sets')

OUT_DIR  = Path('.')
REAL_CSV = OUT_DIR / 'Profiling_Real.csv'
L9_CSV   = OUT_DIR / 'Profiling_L9.csv'

# =========================================================================
# METRIC SETS
# =========================================================================

# Full reduced set — one representative per construct, no within-group redundancy
METRICS_10 = [
    'Total_Cells', 'Min_Cluster_Size', 'Total_CellTypes',
    'Dropout_Pct', 'Median_UMI_Per_Cell',
    'Norm_Shannon_Entropy',
    'Silhouette_Score', 'KNN_Purity',
    'Mean_Pi_Score', 'BCV_Dispersion_Proxy'
]

# Only the four axes directly manipulated in the L9 orthogonal array design
METRICS_4 = [
    'Total_Cells', 'Total_CellTypes',
    'Norm_Shannon_Entropy', 'Mean_Pi_Score'
]

print('Paths resolved:')
print(f'  Real datasets : {REAL_BASE_DIR.resolve()}')
print(f'  L9 datasets   : {L9_BASE_DIR.resolve()}')
print(f'  L9 path exists: {L9_BASE_DIR.exists()}')

In [ ]:
# =========================================================================
# LOADERS
# =========================================================================

def load_real_dataset(dataset_path: Path) -> ad.AnnData:
    """
    Load a real dataset from a 10X-style TSV directory.

    Files: counts.mtx, barcodes.tsv, genes.tsv, metadata.tsv
    """
    counts   = mmread(dataset_path / 'counts.mtx').T.tocsr()
    barcodes = pd.read_csv(
        dataset_path / 'barcodes.tsv', header=None, sep='\t'
    )[0].astype(str).values

    gene_file = dataset_path / 'genes.tsv'
    if not gene_file.exists():
        gene_file = dataset_path / 'features.tsv'
    gene_names = pd.read_csv(gene_file, header=None, sep='\t').iloc[:, 0].astype(str).values

    metadata = pd.read_csv(dataset_path / 'metadata.tsv', index_col=0, sep='\t')

    adata = ad.AnnData(X=counts, obs=metadata)
    adata.obs_names = barcodes
    adata.var_names = gene_names

    # Standardise label column
    if 'Ground_Truth_Celltype' not in adata.obs.columns:
        for alt in ['cell_type', 'celltype', 'CellType', 'label']:
            if alt in adata.obs.columns:
                adata.obs['Ground_Truth_Celltype'] = adata.obs[alt].astype(str)
                break
        else:
            raise KeyError(
                f'No recognised cell type column in {dataset_path}.\n'
                f'Available obs columns: {list(adata.obs.columns)}'
            )

    adata.obs['Ground_Truth_Celltype'] = adata.obs['Ground_Truth_Celltype'].astype(str)
    return adata


def load_l9_replicate(replicate_path: Path) -> ad.AnnData:
    """
    Load one L9 simulation replicate.

    Expected files (matching load_to_anndata.ipynb exactly):
      counts.mtx     — genes x cells (transposed internally to cells x genes)
      genes.txt      — one gene name per line, no header
      cells.txt      — one barcode per line, no header
      metadata.csv   — index col 0; 'cell_type' column used as ground truth
    """
    counts   = mmread(replicate_path / 'counts.mtx').T.tocsr()
    genes    = pd.read_csv(replicate_path / 'genes.txt',    header=None)[0].astype(str).values
    cells    = pd.read_csv(replicate_path / 'cells.txt',    header=None)[0].astype(str).values
    metadata = pd.read_csv(replicate_path / 'metadata.csv', index_col=0)

    adata = ad.AnnData(X=counts, obs=metadata)
    adata.var_names = genes
    adata.obs_names = cells

    # Splatter writes 'cell_type'; standardise to shared column name
    adata.obs['Ground_Truth_Celltype'] = adata.obs['cell_type'].astype(str)
    return adata


# =========================================================================
# DIRECTORY DISCOVERY
# =========================================================================

def discover_real_datasets(base_dir: Path):
    """
    Returns list of (name, path) for every for_use / original subfolder
    that contains a counts.mtx file.
    """
    targets = []
    for study in sorted(base_dir.iterdir()):
        if not study.is_dir():
            continue
        for sub in ['for_use', 'original']:
            p = study / sub
            if p.is_dir() and (p / 'counts.mtx').exists():
                targets.append((f'{study.name}_{sub}', p))
    return targets


def discover_l9_replicates(base_dir: Path):
    """
    Returns list of (name, scenario, path) for every L9 replicate,
    found by globbing for counts.mtx — identical logic to load_to_anndata.ipynb.
    """
    targets = []
    for mtx_file in sorted(base_dir.rglob('counts.mtx')):
        rep_dir  = mtx_file.parent      # e.g. .../S1/rep1
        scenario = rep_dir.parent.name  # 'S1'
        rep      = rep_dir.name         # 'rep1'
        name     = f'{scenario}_{rep}'  # 'S1_rep1'
        targets.append((name, scenario, rep_dir))
    return targets


print('Loaders and discovery functions defined.')

In [ ]:
# =========================================================================
# PROFILING, PCA, AND MATCHING HELPERS
# =========================================================================

def calc_knn_purity(adata, label_col='Ground_Truth_Celltype', n_neighbors=15):
    """
    Mean fraction of k-nearest neighbours sharing the same cell-type label.
    Computed on PCA representation using Euclidean distance.
    Uses the distances graph (exact k neighbours per cell) rather than
    the weighted connectivities graph, ensuring a fixed denominator of k=15.
    """
    assert 'X_pca' in adata.obsm, "Run sc.pp.pca() before calc_knn_purity()"
    sc.pp.neighbors(adata, n_neighbors=n_neighbors, use_rep='X_pca', metric='euclidean')
    dist   = adata.obsp['distances']   # binary sparse: nonzero = exactly k neighbours
    labels = adata.obs[label_col].values

    purities = []
    for i in range(adata.n_obs):
        nbr_idx = dist[i, :].nonzero()[1]
        if len(nbr_idx) == 0:
            continue
        purities.append(np.sum(labels[nbr_idx] == labels[i]) / len(nbr_idx))

    return float(np.mean(purities)) if purities else np.nan


def profile_dataset(adata_raw, dataset_name, label_col='Ground_Truth_Celltype',
                    n_hvgs=2000, n_pcs=30, n_neighbors=15):
    result = {'Sample': dataset_name}

    # --- 1. Label-only metrics -----------------------------------------------
    cluster_counts = adata_raw.obs[label_col].value_counts()
    n_cells        = adata_raw.n_obs
    n_clusters     = len(cluster_counts)
    props          = cluster_counts.values / n_cells

    result['Total_Cells']      = n_cells
    result['Total_CellTypes']  = n_clusters
    result['Min_Cluster_Size'] = int(cluster_counts.min())

    if n_clusters > 1:
        H     = entropy(props, base=np.e)
        H_max = np.log(n_clusters)
        result['Norm_Shannon_Entropy'] = round(H / H_max, 4)
    else:
        result['Norm_Shannon_Entropy'] = np.nan

    # --- 2. Raw-count metrics (compute before any normalisation) -------------
    adata = adata_raw.copy()
    X     = adata.X
    if hasattr(X, 'toarray'):
        n_zeros   = X.shape[0] * X.shape[1] - X.nnz
        lib_sizes = np.asarray(X.sum(axis=1)).flatten()
    else:
        n_zeros   = int(np.sum(X == 0))
        lib_sizes = X.sum(axis=1)

    result['Dropout_Pct']         = round(n_zeros / (adata.n_obs * adata.n_vars) * 100, 2)
    result['Median_UMI_Per_Cell'] = round(float(np.median(lib_sizes)), 1)

    # --- 3. Normalise, then select HVGs with seurat flavour ------------------
    # seurat flavour runs on log-normalised data and always produces
    # dispersions_norm — more robust than seurat_v3 across dataset types.
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=n_hvgs, flavor='seurat')

    hvg_mask = adata.var['highly_variable']

    # BCV proxy: median normalised dispersion of HVGs
    # dispersions_norm is always present with seurat flavour
    result['BCV_Dispersion_Proxy'] = round(
        float(np.median(adata.var.loc[hvg_mask, 'dispersions_norm'])), 4
    )

    # --- 4. Geometry metrics (scale + PCA on HVG log-norm matrix) -----------
    adata_de  = adata[:, hvg_mask].copy()   # pre-scale copy for DE
    adata_geo = adata[:, hvg_mask].copy()

    sc.pp.scale(adata_geo, max_value=10)
    sc.tl.pca(adata_geo, n_comps=min(n_pcs, adata_geo.n_obs - 1, adata_geo.n_vars - 1))

    pca_coords = adata_geo.obsm['X_pca']
    labels_arr = adata_geo.obs[label_col].values

    if n_clusters > 1:
        result['Silhouette_Score'] = round(silhouette_score(pca_coords, labels_arr), 4)
        result['KNN_Purity']       = round(calc_knn_purity(adata_geo, label_col, n_neighbors), 4)
    else:
        result['Silhouette_Score'] = np.nan
        result['KNN_Purity']       = np.nan

    # --- 5. DE difficulty (Pi score) -----------------------------------------
    mean_pi = np.nan
    if n_clusters > 1:
        valid_groups = cluster_counts[cluster_counts >= 3].index.tolist()
        if len(valid_groups) > 1:
            sc.tl.rank_genes_groups(adata_de, label_col, groups=valid_groups,
                                    method='wilcoxon', key_added='de')
            pi_scores = []
            for grp in adata_de.uns['de']['names'].dtype.names:
                pvals = adata_de.uns['de']['pvals_adj'][grp][:50]
                lfcs  = adata_de.uns['de']['logfoldchanges'][grp][:50]
                pi_scores.extend(-np.log10(pvals + 1e-300) * np.abs(lfcs))
            if pi_scores:
                mean_pi = round(float(np.mean(pi_scores)), 4)
    result['Mean_Pi_Score'] = mean_pi

    del adata, adata_de, adata_geo
    gc.collect()
    return result


def run_pca_and_plot(df_real, df_l9_anchors, metric_cols, title_suffix,
                     save_path=None, pca_model=None, scaler=None):
    """
    Standardise → PCA → plot real datasets (grey dots) + L9 anchors (coloured X).

    If scaler and pca_model are provided they are reused (Section 3).
    Otherwise both are fitted on real data.

    Returns (scaler, pca_model, df_pca)
    """
    real_vals = df_real[metric_cols].values
    l9_vals   = df_l9_anchors[metric_cols].values

    if scaler is None:
        scaler = StandardScaler().fit(real_vals)
    if pca_model is None:
        pca_model = PCA(n_components=2).fit(scaler.transform(real_vals))

    real_pca = pca_model.transform(scaler.transform(real_vals))
    l9_pca   = pca_model.transform(scaler.transform(l9_vals))

    df_pca_real = pd.DataFrame({'PC1': real_pca[:, 0], 'PC2': real_pca[:, 1],
                                 'Label': df_real.index, 'Type': 'Real Data'})
    df_pca_l9   = pd.DataFrame({'PC1': l9_pca[:, 0],   'PC2': l9_pca[:, 1],
                                 'Label': df_l9_anchors.index, 'Type': 'L9 Anchor'})
    df_pca = pd.concat([df_pca_real, df_pca_l9], ignore_index=True)

    ev  = pca_model.explained_variance_ratio_
    fig, ax = plt.subplots(figsize=(12, 8))

    sns.scatterplot(data=df_pca[df_pca['Type'] == 'Real Data'],
                    x='PC1', y='PC2', color='lightgray', alpha=0.7,
                    s=50, edgecolor='k', label='Real Data', ax=ax)

    palette = sns.color_palette('tab10', n_colors=len(df_l9_anchors))
    for i, (_, row) in enumerate(df_pca[df_pca['Type'] == 'L9 Anchor'].iterrows()):
        ax.scatter(row['PC1'], row['PC2'], color=palette[i], s=200,
                   marker='X', edgecolor='black', zorder=5, label=row['Label'])

    ax.set_xlabel(f'PC1 ({ev[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({ev[1]*100:.1f}%)')
    ax.set_title(f'Dataset Metric Space — {title_suffix}\n'
                 f'PC1 + PC2 = {(ev[0]+ev[1])*100:.1f}% of total metric variance')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300)
        print(f'  Saved: {save_path}')
    plt.show()
    return scaler, pca_model, df_pca


def assign_scenarios(df_real, df_l9_anchors, metric_cols, scaler):
    """
    Assign each real dataset to its nearest L9 anchor by standardised
    Euclidean distance in the raw metric space (not PCA space).

    Returns DataFrame with:
      Assigned_Scenario, Distance_to_Assigned,
      + distance to every anchor for diagnostics.
    """
    real_sc = scaler.transform(df_real[metric_cols].values)
    l9_sc   = scaler.transform(df_l9_anchors[metric_cols].values)

    df_dist = pd.DataFrame(
        cdist(real_sc, l9_sc, metric='euclidean'),
        index=df_real.index,
        columns=df_l9_anchors.index
    )
    result = pd.concat([
        df_dist.idxmin(axis=1).rename('Assigned_Scenario'),
        df_dist.min(axis=1).round(4).rename('Distance_to_Assigned'),
        df_dist.round(4)
    ], axis=1)
    result.index.name = 'Dataset'
    return result


print('All helpers defined.')

---
## Section 1 — Profile all datasets + 10-metric PCA and matching

This is the only section that reads raw files. Sections 2 and 3 load the saved CSVs.
Profiling the full real + L9 set will take some time — expect 2–5 min per large dataset.

In [ ]:
# =========================================================================
# 1A. PROFILE REAL DATASETS
# =========================================================================
real_targets = discover_real_datasets(REAL_BASE_DIR)
print(f'Found {len(real_targets)} real dataset directories.\n')

real_results = []
for name, path in real_targets:
    print(f'Profiling: {name} ...')
    try:
        adata  = load_real_dataset(path)
        rec    = profile_dataset(adata, name)
        real_results.append(rec)
        print('  ✓ Success')
    except Exception as e:
        print(f'  [!] Failed: {e}')
    finally:
        try:
            del adata
        except NameError:
            pass
        gc.collect()

df_real = pd.DataFrame(real_results).set_index('Sample')
df_real.to_csv(REAL_CSV)
print(f'\nSaved → {REAL_CSV}   shape: {df_real.shape}')
df_real[METRICS_10].head(3)

In [ ]:
# =========================================================================
# 1B. PROFILE L9 REPLICATES
# =========================================================================
l9_targets = discover_l9_replicates(L9_BASE_DIR)
print(f'Found {len(l9_targets)} L9 replicates (expect 27).\n')

l9_results = []
for name, scenario, path in l9_targets:
    print(f'Profiling: {name} ...')
    try:
        adata = load_l9_replicate(path)
        rec   = profile_dataset(adata, name)
        rec['Scenario']  = scenario
        rec['Replicate'] = name.split('_')[-1]   # 'rep1', 'rep2', 'rep3'
        l9_results.append(rec)
        print('  ✓ Success')
    except Exception as e:
        print(f'  [!] Failed: {e}')
    finally:
        try:
            del adata
        except NameError:
            pass
        gc.collect()

df_l9 = pd.DataFrame(l9_results).set_index('Sample')
df_l9.to_csv(L9_CSV)
print(f'\nSaved → {L9_CSV}   shape: {df_l9.shape}')
df_l9[METRICS_10 + ['Scenario', 'Replicate']].head(30)

In [ ]:
# =========================================================================
# 1C. PCA + MATCHING — 10 metrics
# NaN fill uses real-data medians: the scale is defined by real biology,
# not by what Splatter happens to produce.
# =========================================================================
real_10   = df_real[METRICS_10].fillna(df_real[METRICS_10].median())

l9_anchors_10 = (
    df_l9.groupby('Scenario')[METRICS_10].mean()
    .fillna(real_10.median())
)

scaler_10, pca_10, df_pca_10 = run_pca_and_plot(
    real_10, l9_anchors_10, METRICS_10,
    title_suffix='10-Metric Approach',
    save_path=OUT_DIR / 'PCA_10metric.png'
)

mapping_10 = assign_scenarios(real_10, l9_anchors_10, METRICS_10, scaler_10)
mapping_10.to_csv(OUT_DIR / 'Mapping_10metric.csv')

print('\n10-metric assignments:')
print(mapping_10[['Assigned_Scenario', 'Distance_to_Assigned']].to_string())

---
## Section 2 — 4-Axis Approach (L9 design dimensions only)

Matches on the four factors that were directly controlled in the L9 design.
Most causally principled: you are matching on exactly what you manipulated.
Loads from saved CSVs — no re-profiling needed.

In [ ]:
# =========================================================================
# 2A. LOAD + PCA + MATCHING — 4 axes
# =========================================================================
df_real = pd.read_csv(REAL_CSV, index_col=0)
df_l9   = pd.read_csv(L9_CSV,   index_col=0)

real_4 = df_real[METRICS_4].fillna(df_real[METRICS_4].median())

l9_anchors_4 = (
    df_l9.groupby('Scenario')[METRICS_4].mean()
    .fillna(real_4.median())
)

scaler_4, pca_4, df_pca_4 = run_pca_and_plot(
    real_4, l9_anchors_4, METRICS_4,
    title_suffix='4-Axis Approach (L9 Design Dimensions)',
    save_path=OUT_DIR / 'PCA_4axis.png'
)

mapping_4 = assign_scenarios(real_4, l9_anchors_4, METRICS_4, scaler_4)
mapping_4.to_csv(OUT_DIR / 'Mapping_4axis.csv')

print('\n4-axis assignments:')
print(mapping_4[['Assigned_Scenario', 'Distance_to_Assigned']].to_string())

In [ ]:
# =========================================================================
# 2B. DISAGREEMENTS BETWEEN 10-METRIC AND 4-AXIS
# Disagreements indicate the dataset sits between anchors along a dimension
# that the 10-metric set captures but the 4-axis set does not
# (typically separability or sequencing depth).
# =========================================================================
mapping_10 = pd.read_csv(OUT_DIR / 'Mapping_10metric.csv', index_col=0)

compare = pd.DataFrame({
    '10-metric' : mapping_10['Assigned_Scenario'],
    '4-axis'    : mapping_4['Assigned_Scenario'],
})
compare['Agree'] = compare['10-metric'] == compare['4-axis']

n = compare['Agree'].sum()
print(f'Agreement: {n}/{len(compare)} ({n/len(compare)*100:.1f}%)')
if (~compare['Agree']).any():
    print('\nDisagreements (examine these manually):')
    print(compare[~compare['Agree']].to_string())
else:
    print('All assignments agree between the two approaches.')

---
## Section 3 — Real-Data PCA Approach

The scaler and PCA model are fitted **on real datasets only**, then L9 anchors are projected
into that space. Matching happens in the PC space defined by real biological variation.

This is the 'biased toward realness' approach: constructs that co-vary in real biology
are naturally compressed into shared axes rather than given inflated weight by redundant metrics.

Visualisation uses 2 PCs for the figure. Matching uses however many PCs explain ~80%
of real-data variance — checked in cell 3A.

In [ ]:
# =========================================================================
# 3A. FIT SCALER + INSPECT EXPLAINED VARIANCE
# =========================================================================
df_real = pd.read_csv(REAL_CSV, index_col=0)
df_l9   = pd.read_csv(L9_CSV,   index_col=0)

real_10_s3  = df_real[METRICS_10].fillna(df_real[METRICS_10].median())
l9_anchors_s3 = (
    df_l9.groupby('Scenario')[METRICS_10].mean()
    .fillna(real_10_s3.median())
)

# Scale fitted on real data only
scaler_s3   = StandardScaler().fit(real_10_s3.values)
real_sc_s3  = scaler_s3.transform(real_10_s3.values)
l9_sc_s3    = scaler_s3.transform(l9_anchors_s3.values)

# Full PCA to inspect the variance structure
pca_full = PCA().fit(real_sc_s3)
cumvar   = np.cumsum(pca_full.explained_variance_ratio_)
n_pcs_80 = int(np.searchsorted(cumvar, 0.80)) + 1

print('Explained variance per PC (real data only):')
for i, v in enumerate(pca_full.explained_variance_ratio_):
    bar = '█' * int(v * 60)
    print(f'  PC{i+1:2d}: {v*100:5.1f}%  {bar}')
print(f'\nPCs needed for 80% cumulative variance: {n_pcs_80}')
print(f'Matching will use {n_pcs_80} PCs; visualisation uses 2.')

In [ ]:
# =========================================================================
# 3B. 2D VISUALISATION — real-data fitted PCA
# =========================================================================
pca_2d_s3 = PCA(n_components=2).fit(real_sc_s3)

_, _, df_pca_s3 = run_pca_and_plot(
    real_10_s3, l9_anchors_s3, METRICS_10,
    title_suffix='Real-Data PCA (scaler + PCA fitted on real data only)',
    save_path=OUT_DIR / 'PCA_RealDataFitted.png',
    pca_model=pca_2d_s3,
    scaler=scaler_s3
)

In [ ]:
# =========================================================================
# 3C. MATCHING IN n_pcs_80 PC SPACE
# More faithful than 2D: uses the full subspace that explains ~80%
# of real-data variation.
# =========================================================================
pca_match_s3 = PCA(n_components=n_pcs_80).fit(real_sc_s3)
real_pc_s3   = pca_match_s3.transform(real_sc_s3)
l9_pc_s3     = pca_match_s3.transform(l9_sc_s3)

df_dist_s3 = pd.DataFrame(
    cdist(real_pc_s3, l9_pc_s3, metric='euclidean'),
    index=real_10_s3.index,
    columns=l9_anchors_s3.index
)
mapping_s3 = pd.concat([
    df_dist_s3.idxmin(axis=1).rename('Assigned_Scenario'),
    df_dist_s3.min(axis=1).round(4).rename('Distance_to_Assigned'),
    df_dist_s3.round(4)
], axis=1)
mapping_s3.index.name = 'Dataset'
mapping_s3.to_csv(OUT_DIR / 'Mapping_RealDataPCA.csv')

print(f'Matching in {n_pcs_80}-PC space (real-data fitted):')
print(mapping_s3[['Assigned_Scenario', 'Distance_to_Assigned']].to_string())

In [ ]:
# =========================================================================
# 3D-1. HARD CONSCENSUS: THREE-WAY COMPARISON TOP
# Stable = all three approaches agree → high-confidence assignment
# Ambiguous = at least one disagrees → flag these in Methods
# =========================================================================
mapping_10 = pd.read_csv(OUT_DIR / 'Mapping_10metric.csv', index_col=0)
mapping_4  = pd.read_csv(OUT_DIR / 'Mapping_4axis.csv',    index_col=0)

three_way = pd.DataFrame({
    '10-metric'     : mapping_10['Assigned_Scenario'],
    '4-axis'        : mapping_4['Assigned_Scenario'],
    'real-data PCA' : mapping_s3['Assigned_Scenario'],
})
three_way['All_Agree'] = (
    (three_way['10-metric'] == three_way['4-axis']) &
    (three_way['4-axis']    == three_way['real-data PCA'])
)

n_stable = three_way['All_Agree'].sum()
print(f'Stable assignments (all 3 agree)    : {n_stable}/{len(three_way)} '
      f'({n_stable/len(three_way)*100:.1f}%)')
print(f'Ambiguous (any disagreement)        : {len(three_way) - n_stable}')
print()
print(three_way.to_string())

three_way.to_csv(OUT_DIR / 'Mapping_Comparison_AllApproaches.csv')
print('\nSaved: Mapping_Comparison_AllApproaches.csv')

In [ ]:
# =========================================================================
# 3D-2: SOFTER CONSENSUS ASSIGNMENT — Borda count across all three methods
# =========================================================================
# For each method, each real dataset gets a ranked list of L9 scenarios
# by distance. Borda score = 9 - (rank - 1), so rank-1 = 9 pts, rank-9 = 1 pt.
# Summing across methods gives a consensus score per scenario.
# Max possible score = 27 (rank-1 in all three methods).
# =========================================================================

import warnings

def borda_from_distance_csv(csv_path, scenario_cols):
    """Load a mapping CSV and return a Borda score matrix (datasets x scenarios)."""
    df = pd.read_csv(csv_path, index_col=0)
    dist_cols = [c for c in df.columns if c in scenario_cols]
    dist_matrix = df[dist_cols]
    # Rank each row (ascending distance = rank 1 = best)
    ranked = dist_matrix.rank(axis=1, method='min', ascending=True).astype(int)
    n_scenarios = len(dist_cols)
    borda = (n_scenarios + 1) - ranked   # rank-1 → n_scenarios pts
    return borda


# Identify scenario columns (S1..S9)
scenario_cols = [f'S{i}' for i in range(1, 10)]

borda_10  = borda_from_distance_csv(OUT_DIR / 'Mapping_10metric.csv',    scenario_cols)
borda_4   = borda_from_distance_csv(OUT_DIR / 'Mapping_4axis.csv',       scenario_cols)
borda_pca = borda_from_distance_csv(OUT_DIR / 'Mapping_RealDataPCA.csv', scenario_cols)

# Sum Borda scores
borda_total = borda_10.add(borda_4, fill_value=0).add(borda_pca, fill_value=0)
borda_total.index.name = 'Dataset'

# Consensus assignment = scenario with highest total Borda score
consensus_scenario = borda_total.idxmax(axis=1).rename('Consensus_Scenario')
consensus_score    = borda_total.max(axis=1).rename('Consensus_Borda_Score')

# Flag confidence level
# Max possible = 27 (rank-1 in all 3 methods)
# ≥ 24 = strong (rank-1 or rank-2 in all methods)
# 18–23 = moderate
# < 18 = weak / ambiguous
def confidence(score):
    if score >= 24:
        return 'strong'
    elif score >= 18:
        return 'moderate'
    else:
        return 'weak'

consensus_df = pd.concat([
    consensus_scenario,
    consensus_score,
    borda_total
], axis=1)

consensus_df['Confidence'] = consensus_df['Consensus_Borda_Score'].apply(confidence)

# Also show the runner-up scenario and its score
runner_up_scenario = borda_total.apply(lambda row: row.nlargest(2).index[-1], axis=1)
runner_up_score    = borda_total.apply(lambda row: row.nlargest(2).iloc[-1],  axis=1)
consensus_df['Runner_Up_Scenario'] = runner_up_scenario
consensus_df['Runner_Up_Score']    = runner_up_score
consensus_df['Score_Gap']          = consensus_score - runner_up_score

# =========================================================================
# SORTED CONSENSUS TABLE — ranked by score then gap, grouped by scenario
# =========================================================================

# Sort: primary = Consensus_Borda_Score desc, secondary = Score_Gap desc
consensus_sorted = (
    consensus_df[['Consensus_Scenario', 'Consensus_Borda_Score',
                  'Runner_Up_Scenario', 'Score_Gap', 'Confidence']]
    .sort_values(['Consensus_Scenario',
                  'Consensus_Borda_Score',
                  'Score_Gap'],
                 ascending=[True, False, False])
)

print('Consensus assignments (Borda count, max score = 27) — grouped by L9 scenario, best matches first\n')
print()
print(f'{"Score":>6}  {"Gap":>5}  {"Conf":<8}  {"Runner-up":<6}  Dataset')
print('─' * 80)

current_scenario = None
for dataset, row in consensus_sorted.iterrows():
    scen = row['Consensus_Scenario']

    # Print scenario header when group changes
    if scen != current_scenario:
        if current_scenario is not None:
            print()
        print(f'\n── {scen} ─────────────────────────────────────────')
        current_scenario = scen

    conf_symbol = {'strong': '✅', 'moderate': '⚠️ ', 'weak': '❌'}.get(row['Confidence'], '?')
    print(f"  {row['Consensus_Borda_Score']:>4.0f}   {row['Score_Gap']:>4.0f}  "
          f"{conf_symbol}        {row['Runner_Up_Scenario']:<6}  {dataset}")

print('\n' + '─' * 80)
print(f"✅ strong (score ≥ 24, gap ≥ 8)   ⚠️  moderate   ❌ weak / ambiguous")

consensus_sorted.to_csv(OUT_DIR / 'Mapping_Consensus_Borda_Sorted.csv')
print(f'\nSaved: Mapping_Borda_Consensus_Sorted.csv')

In [ ]:
# =========================================================================
# 3E. PC LOADINGS — what drives variation among real datasets
# Reportable finding: if Norm_Shannon_Entropy dominates PC1 and
# Mean_Pi_Score barely loads, real datasets vary more in class balance
# than in DE signal — worth one sentence in Methods or Supplementary.
# =========================================================================
loadings = pd.DataFrame(
    pca_2d_s3.components_.T,
    index=METRICS_10,
    columns=['PC1', 'PC2']
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, pc in zip(axes, ['PC1', 'PC2']):
    loadings[pc].sort_values().plot(kind='barh', ax=ax,
                                    color='steelblue', edgecolor='k')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'{pc} Loadings  —  Real-data fitted PCA')
    ax.set_xlabel('Loading coefficient')

plt.tight_layout()
plt.savefig(OUT_DIR / 'PCA_RealData_Loadings.png', dpi=300)
plt.show()
print('Saved: PCA_RealData_Loadings.png')